# Do binder-only vs non-binder-only pose VAEs learn different latent spaces?

Tests whether a VAE trained on **binder** poses and a VAE trained on **non-binder**
poses learn different manifolds, and whether that difference is binding-relevant.

For honesty, the two VAEs are (by default) **re-pretrained on a TRAIN split** and all
metrics are computed on a held-out TEST split (recon error on training poses is
optimistically low). Set `USE_SAVED=True` to instead load the saved full-corpus VAEs
(faster, but recon error is then leaky — interpret with care).

Outputs: 2x2 cross-reconstruction matrix, Δ-recon AUROC, latent CKA, per-latent probe
AUROC, and UMAP/PCA plots. Runtime → GPU.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import sys, os
REPO="/content/drive/MyDrive/tcrpmhc_pose_binding"; DATA_DIR=f"{REPO}/data"
PRETRAIN_DIR=f"{REPO}/pretrained"; OUT_DIR=f"{REPO}/results"; os.makedirs(OUT_DIR,exist_ok=True)
sys.path.insert(0,f"{REPO}/src")
import subprocess
subprocess.run([sys.executable,"-m","pip","install","-q","-r",f"{REPO}/requirements.txt"])
subprocess.run([sys.executable,"-m","pip","install","-q","umap-learn"])
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from data import load_data, build_pose, cluster_tcrs
from train_utils import pretrain_posevae, DEVICE
from pose_vae import PoseVAERaw
from analysis import recon_error_per_sample, latent_mu, linear_cka
print("device:",DEVICE)

## Config + data + held-out split

In [ ]:
USE_SAVED=False      # True -> load saved full-corpus VAEs (leaky); False -> re-pretrain on TRAIN
ROT="6d"; PRETRAIN_EPOCHS=40
BINDER_SAVED=f"{PRETRAIN_DIR}/posevae_binderOnly_iptm05_{ROT}.pt"
NONBINDER_SAVED=f"{PRETRAIN_DIR}/posevae_nonbinderOnly_iptm05_{ROT}.pt"   # must exist if USE_SAVED

D=load_data(DATA_DIR, min_iptm=0.5); pool=D["pool"]; ts=D["trans_scale"]
X=build_pose(pool, ts); y=pool.label.to_numpy().astype(int); g=cluster_tcrs(pool)
tr,te=next(iter(StratifiedGroupKFold(5,shuffle=True,random_state=42).split(np.zeros(len(y)),y,g)))
Xtr,Xte,ytr,yte=X[tr],X[te],y[tr],y[te]
print("test:",len(te),"| test binders:",int(yte.sum()),"nonbinders:",int((yte==0).sum()))

## Get the two VAEs (re-pretrain on TRAIN, or load saved)

In [ ]:
def load_vae(path):
    v=PoseVAERaw(rotation_encoder=ROT).to(DEVICE); v.load_state_dict(torch.load(path,map_location=DEVICE)); v.eval(); return v
if USE_SAVED:
    vae_b=load_vae(BINDER_SAVED); vae_n=load_vae(NONBINDER_SAVED); print("loaded saved VAEs")
else:
    vae_b=pretrain_posevae(Xtr[ytr==1], ROT, epochs=PRETRAIN_EPOCHS)   # binder-only (train)
    vae_n=pretrain_posevae(Xtr[ytr==0], ROT, epochs=PRETRAIN_EPOCHS)   # nonbinder-only (train)
    print("re-pretrained on train split | binders:",int((ytr==1).sum()),"nonbinders:",int((ytr==0).sum()))

## 1. Cross-reconstruction matrix (held-out)

In [ ]:
eb=recon_error_per_sample(vae_b,Xte,DEVICE); en=recon_error_per_sample(vae_n,Xte,DEVICE)
mb=yte==1; mn=yte==0
mat=pd.DataFrame({"VAE_binder":[eb[mb].mean(),eb[mn].mean()],
                 "VAE_nonbinder":[en[mb].mean(),en[mn].mean()]}, index=["binder poses","nonbinder poses"])
print("mean reconstruction error (lower=better):\n", mat.round(4))
print("\nMatched < mismatched?  binder:", eb[mb].mean()<en[mb].mean(), " nonbinder:", en[mn].mean()<eb[mn].mean())

## 2. Δ-reconstruction as a binder score

In [ ]:
delta = en - eb     # >0 => binder-VAE reconstructs better => predict binder
auc=roc_auc_score(yte, delta)
print(f"Delta-recon AUROC (binder vs nonbinder): {auc:.3f}")
fig,ax=plt.subplots(figsize=(6,4))
ax.hist(delta[mb],bins=40,alpha=0.6,label="binder",density=True)
ax.hist(delta[mn],bins=40,alpha=0.6,label="nonbinder",density=True)
ax.axvline(0,c="k",lw=1); ax.set_xlabel("recon(VAE_nonbinder) - recon(VAE_binder)"); ax.set_title(f"Delta-recon  (AUROC={auc:.3f})"); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/latent_delta_recon.png",dpi=150); plt.show()

## 3. Latent CKA + per-latent probe AUROC

In [ ]:
Zb=latent_mu(vae_b,Xte,DEVICE); Zn=latent_mu(vae_n,Xte,DEVICE)
print(f"linear CKA(Z_binder, Z_nonbinder) = {linear_cka(Zb,Zn):.3f}   (1=identical reps)")
def probe_auc(Z,y):
    p=cross_val_predict(make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,class_weight='balanced')),
                        Z,y,cv=5,method='predict_proba')[:,1]
    return roc_auc_score(y,p)
print(f"probe AUROC on Z_binder    = {probe_auc(Zb,yte):.3f}")
print(f"probe AUROC on Z_nonbinder = {probe_auc(Zn,yte):.3f}")

## 4. UMAP of the two latent spaces (colored by label)

In [ ]:
try:
    import umap; red=lambda Z: umap.UMAP(n_neighbors=15,min_dist=0.1,random_state=0).fit_transform(Z); kind="UMAP"
except Exception:
    from sklearn.decomposition import PCA; red=lambda Z: PCA(2).fit_transform(Z); kind="PCA"
fig,ax=plt.subplots(1,2,figsize=(13,5.2))
for a,(Z,name) in zip(ax,[(Zb,"Z_binder (VAE_binder)"),(Zn,"Z_nonbinder (VAE_nonbinder)")]):
    emb=red(Z)
    a.scatter(emb[mn,0],emb[mn,1],s=6,alpha=0.4,label="nonbinder",c="#4C72B0")
    a.scatter(emb[mb,0],emb[mb,1],s=6,alpha=0.6,label="binder",c="#C44E52")
    a.set_title(f"{kind}: {name}"); a.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/latent_umap.png",dpi=150); plt.show()
print("saved figures to", OUT_DIR)